# `fastcdp` Route Interception & Dialog Harness

> Reusable `fastcdp` async context manager (`open_solveit`), CDP `Fetch` HTML/asset interceptor (`install_craft_routes`), health probe (`inspect_dialog`), and smoke checks (`smoke_dialog`) for testing local CRAFT changes safely against any Solveit dialog.

In [ ]:
#| default_exp cdp

In [ ]:
#| export
import asyncio, base64, mimetypes, tempfile
from contextlib import asynccontextmanager
from dataclasses import dataclass, field
from urllib.parse import urlparse
from fastcore.utils import *

from craft_dev.auth import (
    DEFAULT_AUTH_FILE, REPO_ROOT, alogin, cookies_for_cdp, default_target_url,
    ensure_auth, is_authenticated_page, resolve_browser_executable,
)

In [ ]:
#| export
ROOT_ASSET_NAMES = {'CRAFT.js', 'CRAFT.css'}
_CONTENT_TYPES = {'.js': 'application/javascript; charset=utf-8', '.css': 'text/css; charset=utf-8', '.svg': 'image/svg+xml'}
_CORE_GLOBALS = ('craftC2', 'craftOmit', 'craftParts', 'craftActionMenus', 'craftCellChains', 'solveitHotkeys')
DESKTOP_METRICS = dict(width=1440, height=900, deviceScaleFactor=1, mobile=False)
MOBILE_METRICS = dict(width=390, height=844, deviceScaleFactor=3, mobile=True)

@dataclass
class InterceptRecord:
    url: str
    action: str  # "served-local" | "blocked-vanilla" | "passthrough"
    local_path: str | None = None

@dataclass
class InspectReport:
    url: str
    final_url: str
    mode: str
    assets_dir: str
    authenticated: bool
    title: str
    intercepted: list[InterceptRecord] = field(default_factory=list)
    console_errors: list[str] = field(default_factory=list)
    console_warnings: list[str] = field(default_factory=list)
    page_errors: list[str] = field(default_factory=list)
    failed_requests: list[dict] = field(default_factory=list)
    craft_health: dict = field(default_factory=dict)
    eval_result: object = None
    screenshot: str | None = None

HEALTH_PROBE_JS = r"""(() => {
  const globals = [
    'craftC2', 'craftMd', 'craftInlineNote', 'craftInlineText', 'craftOmit', 'craftParts',
    'craftActionMenus', 'craftCellChains', 'craftIdleComposer', 'craftInsertPreview',
    'craftImageDrop', 'craftMobile', 'craftMobileBar', 'solveitHotkeys', '__hotkeyRegister',
  ];
  const globalStatus = {};
  for (const name of globals) {
    const val = window[name];
    globalStatus[name] = val == null ? null : typeof val;
  }
  const cells = [...document.querySelectorAll('#dialog-container > .editable')];
  const cellCounts = {};
  for (const c of cells) {
    const mtype = c.querySelector('.card-in')?.dataset?.mtype || c.dataset?.mtype || 'unknown';
    cellCounts[mtype] = (cellCounts[mtype] || 0) + 1;
  }
  const cs = getComputedStyle(document.documentElement);
  const cssVars = {
    '--craft-prose': cs.getPropertyValue('--craft-prose').trim(),
    '--craft-code': cs.getPropertyValue('--craft-code').trim(),
    '--craft-prose-family': cs.getPropertyValue('--craft-prose-family').trim(),
    '--craft-code-family': cs.getPropertyValue('--craft-code-family').trim(),
  };
  const craftTags = [...document.querySelectorAll('script[src], link[href]')]
    .map(el => el.getAttribute('src') || el.getAttribute('href'))
    .filter(u => u && (/CRAFT\.(js|css)/i.test(u) || u.includes('.craft-assets')));
  return {
    hasDialogContainer: !!document.getElementById('dialog-container'),
    hasFileTreePanel: !!document.getElementById('solveit-file-tree-panel'),
    hasLauncher: !!document.getElementById('solveit-launcher'),
    isMobile: document.documentElement.classList.contains('craft-mobile'),
    totalCells: cells.length, cellCounts,
    hotkeyCount: window.solveitHotkeys?.list?.()?.length ?? null,
    globals: globalStatus, cssVars, craftTags,
  };
})()"""

## Local Asset Resolution & Inline HTML Patching

Solveit serves `.craft-assets/*` as static files and **inlines** `CRAFT.css` (`<style>`) and `CRAFT.js` (`<script type="module">`) directly into `/dialog_` HTML responses. `install_craft_routes` intercepts both using the CDP `Fetch` domain.

In [ ]:
#| export
def resolve_local_asset(url_path, assets_dir):
    "Resolve a Solveit asset `url_path` to a local file in `assets_dir`, if present."
    parts = [p for p in url_path.split('?', 1)[0].split('/') if p]
    if not parts: return None
    if '.craft-assets' in parts:
        rel = Path(*parts[parts.index('.craft-assets') + 1:])
        return first(c for c in (assets_dir/'.craft-assets'/rel, assets_dir/rel) if c.is_file())
    if parts[-1] in ROOT_ASSET_NAMES and (c := assets_dir/parts[-1]).is_file(): return c
    return None

def guess_content_type(path):
    return _CONTENT_TYPES.get(path.suffix.lower()) or mimetypes.guess_type(str(path))[0] or 'application/octet-stream' 

In [ ]:
import tempfile
from fastcore.test import test_eq

`resolve_local_asset` maps both root `CRAFT.js`/`CRAFT.css` requests and `.craft-assets/*` paths to local files, ignoring query strings.

In [ ]:
test_eq(resolve_local_asset('/static/CRAFT.js?v=1', REPO_ROOT), REPO_ROOT/'CRAFT.js')
test_eq(resolve_local_asset('/static/.craft-assets/craft_md.js', REPO_ROOT), REPO_ROOT/'craft_md.js')
test_eq(resolve_local_asset('/static/unrelated.js', REPO_ROOT), None)
test_eq(guess_content_type(Path('CRAFT.js')), 'application/javascript; charset=utf-8')

In [ ]:
#| export
_STYLE_RE = re.compile(r'(<style\b[^>]*>)([\s\S]*?)(</style>)', re.I)
_SCRIPT_RE = re.compile(r'(<script\b[^>]*>)([\s\S]*?)(</script>)', re.I)

def _patch_block(html, rx, pred, path, label, anchor, open_tag, close_tag, mode, records):
    replaced = False
    def repl(m):
        nonlocal replaced
        if not pred(m.group(2)): return m.group(0)
        replaced = True
        if mode == 'vanilla':
            records.append(InterceptRecord(url=label, action='blocked-vanilla'))
            return ''
        if path.is_file():
            records.append(InterceptRecord(url=label, action='served-local', local_path=str(path)))
            return f"{m.group(1)}\n{path.read_text(encoding='utf-8')}\n{m.group(3)}"
        return m.group(0)
    patched = rx.sub(repl, html)
    if mode == 'local' and not replaced and path.is_file() and anchor in patched:
        records.append(InterceptRecord(url=f"{label} (injected)", action='served-local', local_path=str(path)))
        patched = patched.replace(anchor, f"{open_tag}\n{path.read_text(encoding='utf-8')}\n{close_tag}\n{anchor}", 1)
    return patched

def patch_dialog_html(html, *, mode, assets_dir):
    "Replace or strip Solveit's inlined `CRAFT.css` `<style>` and `CRAFT.js` `<script>` blocks."
    records = []
    if mode == 'live': return html, records
    html = _patch_block(html, _STYLE_RE, lambda b: '--craft-prose' in b or 'Tokyo Night Pure' in b,
                        assets_dir/'CRAFT.css', 'inline:<style>CRAFT.css', '</head>', '<style>', '</style>', mode, records)
    html = _patch_block(html, _SCRIPT_RE, lambda b: 'craftC2' in b and 'hotkey_manager.js' in b,
                        assets_dir/'CRAFT.js', 'inline:<script>CRAFT.js', '</body>', '<script type="module">', '</script>', mode, records)
    return html, records

In `local` mode `patch_dialog_html` swaps inlined CRAFT blocks with local files (injecting them if absent); in `vanilla` mode it strips them.

In [ ]:
with tempfile.TemporaryDirectory() as d:
    td = Path(d)
    (td/'CRAFT.css').write_text('/* LOCAL CSS */')
    (td/'CRAFT.js').write_text('/* LOCAL JS */')
    html = '<html><head><style>:root{--craft-prose:11px;}</style></head><body><script type="module">// hotkey_manager.js craftC2</script></body></html>'
    patched, recs = patch_dialog_html(html, mode='local', assets_dir=td)
    test_eq(('/* LOCAL CSS */' in patched, '/* LOCAL JS */' in patched, [r.action for r in recs]), (True, True, ['served-local', 'served-local']))
    vanilla, vrecs = patch_dialog_html(html, mode='vanilla', assets_dir=td)
    test_eq(('--craft-prose' in vanilla, 'craftC2' in vanilla, [r.action for r in vrecs]), (False, False, ['blocked-vanilla', 'blocked-vanilla']))

In [ ]:
#| export
_FETCH_PATTERNS = [
    {'urlPattern': '*/dialog*', 'resourceType': 'Document', 'requestStage': 'Response'},
    {'urlPattern': '*.craft-assets/*', 'requestStage': 'Request'},
    {'urlPattern': '*/CRAFT.js*', 'requestStage': 'Request'},
    {'urlPattern': '*/CRAFT.css*', 'requestStage': 'Request'},
]

async def _handle_paused(page, p, *, mode, assets_dir, records):
    rid, url = p['requestId'], p['request']['url']
    is_resp = 'responseStatusCode' in p
    try:
        if is_resp:
            code, hdrs = p['responseStatusCode'], p.get('responseHeaders') or []
            ctype = next((h['value'] for h in hdrs if h['name'].lower() == 'content-type'), '')
            if 200 <= code < 300 and 'text/html' in ctype.lower():
                res = await page.Fetch.getResponseBody(requestId=rid)
                raw = base64.b64decode(res['body']).decode('utf-8', 'replace') if res.get('base64Encoded') else res['body']
                patched, doc_recs = patch_dialog_html(raw, mode=mode, assets_dir=assets_dir)
                records.extend(doc_recs)
                out_hdrs = [h for h in hdrs if h['name'].lower() != 'content-length']
                b64 = base64.b64encode(patched.encode('utf-8')).decode('ascii')
                await page.Fetch.fulfillRequest(requestId=rid, responseCode=code, responseHeaders=out_hdrs, body=b64)
                return
            await page.Fetch.continueResponse(requestId=rid)
            return

        path = urlparse(url).path
        if mode == 'vanilla':
            ctype = 'text/css' if path.endswith('.css') else 'application/javascript'
            records.append(InterceptRecord(url=url, action='blocked-vanilla'))
            await page.Fetch.fulfillRequest(requestId=rid, responseCode=200,
                                            responseHeaders=[{'name': 'Content-Type', 'value': ctype}], body='')
            return
        if (local_file := resolve_local_asset(path, assets_dir)) is not None:
            records.append(InterceptRecord(url=url, action='served-local', local_path=str(local_file)))
            b64 = base64.b64encode(local_file.read_bytes()).decode('ascii')
            await page.Fetch.fulfillRequest(requestId=rid, responseCode=200,
                                            responseHeaders=[{'name': 'Content-Type', 'value': guess_content_type(local_file)}], body=b64)
            return
        records.append(InterceptRecord(url=url, action='passthrough'))
        await page.Fetch.continueRequest(requestId=rid)
    except Exception:
        try:
            if is_resp: await page.Fetch.continueResponse(requestId=rid)
            else: await page.Fetch.continueRequest(requestId=rid)
        except Exception: pass

@asynccontextmanager
async def install_craft_routes(page, *, mode='local', assets_dir=REPO_ROOT):
    "Enable CDP `Fetch` interception for CRAFT assets and inlined dialog `CRAFT.js`/`CRAFT.css`."
    records = []
    if mode == 'live':
        yield records
        return
    async with page.cdp.on('Fetch.requestPaused') as q:
        await page.Fetch.enable(patterns=_FETCH_PATTERNS)
        async def _pump():
            while True:
                m = await q.get()
                if m.get('sessionId') == page.sid:
                    await _handle_paused(page, m['params'], mode=mode, assets_dir=assets_dir, records=records)
        task = asyncio.create_task(_pump())
        try: yield records
        finally:
            task.cancel()
            try: await task
            except asyncio.CancelledError: pass
            try: await page.Fetch.disable()
            except Exception: pass

async def click_at(page, x, y):
    "Dispatch a mouse click at viewport coordinates `(x, y)`."
    for kind in ('mouseMoved', 'mousePressed', 'mouseReleased'):
        kw = dict(button='left', clickCount=1) if kind != 'mouseMoved' else {}
        await page.Input.dispatchMouseEvent(type=kind, x=x, y=y, **kw)

## Reusable Async Context Manager & High-Level Helpers

Use `async with open_solveit() as (page, intercepted, page_errors):` from any async test, script, or notebook in this repository to get an authenticated `fastcdp` `Page` with local CRAFT overrides already wired up.

In [ ]:
#| export
def default_assets_dir():
    if env := os.environ.get('CRAFT_ASSETS_DIR'): return Path(env).resolve()
    return (REPO_ROOT/'dist' if (REPO_ROOT/'dist'/'CRAFT.js').is_file() else REPO_ROOT).resolve()

def default_craft_mode(): return os.environ.get('CRAFT_TEST_MODE', 'local')

def _fmt_exception(params):
    d = params.get('exceptionDetails') or {}
    return nested_idx(d, 'exception', 'description') or d.get('text') or str(d)

async def _open_session(target_url, auth_path, *, mode, assets_dir, wait_ms, headed, exec_path, mobile):
    from fastcdp import CDP

    td = tempfile.TemporaryDirectory(prefix='craft-cdp-')
    cdp = await CDP.launch(user_data_dir=td.name, headless=not headed, reuse=False, chrome=exec_path)
    try:
        if cookies := cookies_for_cdp(auth_path): await cdp.Storage.setCookies(cookies=cookies)
        page = await cdp.new_page()
        await page.Emulation.setDeviceMetricsOverride(**(MOBILE_METRICS if mobile else DESKTOP_METRICS))
        if mobile: await page.Emulation.setTouchEmulationEnabled(enabled=True)
        await page.start_console()
        page_errors = []
        exc_cm = cdp.on('Runtime.exceptionThrown')
        exc_q = await exc_cm.__aenter__()
        async def _watch_exc():
            while True:
                m = await exc_q.get()
                if m.get('sessionId') == page.sid: page_errors.append(_fmt_exception(m['params']))
        exc_task = asyncio.create_task(_watch_exc())
        route_cm = install_craft_routes(page, mode=mode, assets_dir=assets_dir)
        records = await route_cm.__aenter__()
        await page.goto(target_url, wait='load', timeout=30)
        if wait_ms: await asyncio.sleep(wait_ms / 1000)
        return td, cdp, page, records, page_errors, route_cm, exc_cm, exc_task
    except BaseException:
        await cdp.quit()
        td.cleanup()
        raise

async def _close_session(td, cdp, route_cm, exc_cm, exc_task):
    exc_task.cancel()
    try: await exc_task
    except asyncio.CancelledError: pass
    try: await exc_cm.__aexit__(None, None, None)
    except Exception: pass
    try: await route_cm.__aexit__(None, None, None)
    except Exception: pass
    try: await cdp.quit()
    finally: td.cleanup()

@asynccontextmanager
async def open_solveit(
    url=None, # Target Solveit dialog URL (defaults to `SOLVEIT_TEST_URL` or saved config)
    *,
    mode=None, # Asset mode: `'local'`, `'live'`, or `'vanilla'` (defaults to `CRAFT_TEST_MODE` or `'local'`)
    assets_dir=None, # Local CRAFT assets directory (defaults to `CRAFT_ASSETS_DIR`, `dist/`, or repo root)
    auth_file=DEFAULT_AUTH_FILE, # Saved cookie JSON path
    wait_ms=2000, # Milliseconds to wait after `load`
    headed=False, # Show browser window instead of headless mode
    browser_name='chrome', # `'chrome'`, `'chromium'`, or `'vivaldi'`
    executable_path=None, # Explicit browser binary path
    auto_auth=False, # Trigger Chrome login automatically if session is missing or expired
    mobile=False, # Emulate a mobile touch viewport (390×844)
):
    "Open an authenticated Solveit page via `fastcdp` with CRAFT route interception, yielding `(page, records, page_errors)`."
    target_url = url or default_target_url()
    if not target_url: raise ValueError("Pass url=... or run `craft-cdp auth --url ...` first to save a default URL.")
    auth_path = ensure_auth(auth_file, auto_login=auto_auth, url=target_url)
    exec_path = resolve_browser_executable(browser_name, executable_path)
    mode = mode or default_craft_mode()
    assets_dir = Path(assets_dir).resolve() if assets_dir else default_assets_dir()
    kw = dict(mode=mode, assets_dir=assets_dir, wait_ms=wait_ms, headed=headed, exec_path=exec_path, mobile=mobile)

    sess = await _open_session(target_url, auth_path, **kw)
    if not await is_authenticated_page(sess[2]) and auto_auth:
        await _close_session(sess[0], sess[1], sess[5], sess[6], sess[7])
        await alogin(url=target_url, auth_file=auth_path)
        sess = await _open_session(target_url, auth_path, **kw)
    td, cdp, page, records, page_errors, route_cm, exc_cm, exc_task = sess
    try:
        if not await is_authenticated_page(page):
            cur = await page.eval('location.href')
            raise PermissionError(
                f"Session invalid or expired at {cur}. Run `.venv/bin/craft-cdp auth` to refresh.")
        yield page, records, page_errors
    finally:
        await _close_session(td, cdp, route_cm, exc_cm, exc_task)

In [ ]:
#| export
async def ainspect_dialog(
    *, url=None, auth_file=DEFAULT_AUTH_FILE, mode='local', assets_dir=REPO_ROOT, wait_ms=2000,
    eval_js=None, screenshot=None, headed=False, browser_name='chrome', executable_path=None, mobile=False,
):
    "Load a Solveit page via `fastcdp` and return a structured `InspectReport`."
    target_url = url or default_target_url()
    if not target_url: raise SystemExit("Pass --url or run `craft-cdp auth --url ...` first.")
    auth_path = ensure_auth(auth_file)
    exec_path = resolve_browser_executable(browser_name, executable_path)
    assets_dir = Path(assets_dir).resolve()

    sess = await _open_session(target_url, auth_path, mode=mode, assets_dir=assets_dir,
                               wait_ms=wait_ms, headed=headed, exec_path=exec_path, mobile=mobile)
    td, cdp, page, records, page_errors, route_cm, exc_cm, exc_task = sess
    try:
        final_url = await page.eval('location.href') or target_url
        authenticated = await is_authenticated_page(page)
        title = await page.eval('document.title') or ''
        health = await page.eval(HEALTH_PROBE_JS) if authenticated else {}
        eval_result = await page.eval(eval_js) if eval_js and authenticated else None
        msgs = await page.console()
        console_errors = [m for m in msgs if m.startswith('error:')]
        console_warnings = [m for m in msgs if m.startswith('warning:')]
        if screenshot:
            p = Path(screenshot)
            p.parent.mkdir(parents=True, exist_ok=True)
            p.write_bytes(await page.screenshot())
    finally:
        await _close_session(td, cdp, route_cm, exc_cm, exc_task)

    return InspectReport(
        url=target_url, final_url=final_url, mode=mode, assets_dir=str(assets_dir),
        authenticated=authenticated, title=title, intercepted=records,
        console_errors=console_errors, console_warnings=console_warnings,
        page_errors=page_errors, craft_health=health, eval_result=eval_result,
        screenshot=str(screenshot) if screenshot else None)

def inspect_dialog(**kwargs):
    "Synchronous wrapper around `ainspect_dialog`."
    return asyncio.run(ainspect_dialog(**kwargs))

async def asmoke_dialog(
    *, url=None, auth_file=DEFAULT_AUTH_FILE, mode='local', assets_dir=REPO_ROOT, wait_ms=2000,
    screenshot_dir=None, headed=False, browser_name='chrome', executable_path=None, mobile=False,
):
    "Run non-destructive UI smoke checks against a Solveit dialog via `fastcdp`."
    checks = []
    def record(name, ok, detail=''): checks.append((name, ok, detail))

    try:
        async with open_solveit(
            url=url, mode=mode, assets_dir=assets_dir, auth_file=auth_file, wait_ms=wait_ms,
            headed=headed, browser_name=browser_name, executable_path=executable_path, mobile=mobile,
        ) as (page, intercepted, page_errors):
            record('authenticated', True, await page.eval('location.href'))
            health = await page.eval(HEALTH_PROBE_JS)
            record('dialog-container-present', bool(health['hasDialogContainer']))
            if mode == 'local':
                served = [r for r in intercepted if r.action == 'served-local']
                record('local-assets-intercepted', bool(served), f"{len(served)} files")
            if mode != 'vanilla':
                missing = [g for g in _CORE_GLOBALS if health['globals'].get(g) is None]
                record('core-craft-globals', not missing, f"missing={missing}" if missing else 'all present')
                for name, open_kw, close_kw, sel in [
                    ('file-tree-toggle', dict(key='b', mod=True), dict(key='b', mod=True), '#solveit-file-tree-panel.open'),
                    ('launcher-toggle', dict(key='P', mod=True, shift=True), dict(key='Escape'), '#solveit-launcher.open'),
                ]:
                    await page.press(**open_kw)
                    try: await page.wait_for_selector(sel, timeout=3); opened = True
                    except Exception: opened = False
                    await page.press(**close_kw)
                    try: await page.wait_for_selector(sel, present=False, timeout=3); closed = True
                    except Exception: closed = False
                    record(name, opened and closed, f"open={opened}, closed={closed}")
            record('no-uncaught-page-errors', not page_errors, '; '.join(page_errors[:3]))
            if screenshot_dir:
                out = Path(screenshot_dir)
                out.mkdir(parents=True, exist_ok=True)
                (out/f'smoke-{mode}.png').write_bytes(await page.screenshot())
    except PermissionError as exc:
        record('authenticated', False, str(exc))
    return checks

def smoke_dialog(**kwargs):
    "Synchronous wrapper around `asmoke_dialog`."
    return asyncio.run(asmoke_dialog(**kwargs))

## Live `fastcdp` Bootstrap & Vanilla Mode Verification

Run with `.venv/bin/nbdev-test --flags live` against an authenticated Solveit dialog.

In [ ]:
#| live
import json
from fastcdp import Rungs

CORE_GLOBALS = (
    'craftC2', 'craftMd', 'craftOmit', 'craftParts', 'craftActionMenus',
    'craftCellChains', 'craftIdleComposer', 'craftInsertPreview', 'craftImageDrop',
    'craftMobile', 'craftMobileBar', 'solveitHotkeys', 'solveitFileTree',
)

async with open_solveit() as (page, intercepted, page_errors):
    assert not page_errors, f"Uncaught page errors: {page_errors}"
    health = await page.eval(HEALTH_PROBE_JS)
    assert health['hasDialogContainer']
    assert all(health['cssVars'].get(k) for k in ('--craft-prose', '--craft-code', '--craft-prose-family', '--craft-code-family'))
    missing = await page.eval(f"{json.dumps(list(CORE_GLOBALS))}.filter(n => window[n] == null)")
    assert not missing, f"Missing CRAFT globals: {missing}"
    if default_craft_mode() == 'local':
        served = {r.url for r in intercepted if r.action == 'served-local'}
        for asset in ('CRAFT.css', 'CRAFT.js', 'craft_md.js', 'hotkey_manager.js'):
            assert any(asset in u for u in served), f"Expected local interception for {asset}"

async with open_solveit(mode='vanilla') as (page, intercepted, page_errors):
    assert not page_errors, f"Uncaught page errors in vanilla mode: {page_errors}"
    assert await page.sel_exists('#dialog-container')
    assert not any(r.action == 'served-local' for r in intercepted)
    assert await page.eval("window.craftC2 === undefined && window.craftMd === undefined && window.craftMobile === undefined")